# SciGeoGuard-X — Final Blinded Human Validation v7

## 60-case stratified human audit with two independent reviewers, blind adjudication, and final system comparison

This notebook is the human-validation stage after the v3 controlled benchmark and v6 strong 7B LLM baselines.

### Core principle

The human reviewers must **not** see:

- SciGeoGuard-X verdicts;
- Qwen-7B verdicts;
- Mistral-7B verdicts;
- generated benchmark labels;
- benchmark family/error names;
- original case IDs.

The original benchmark IDs contain terms such as `angular_area`, `categorical_interpolation`, and `future_information`, which could reveal the intended error family. v7 therefore replaces them with neutral IDs such as:

`HV60_001`, `HV60_002`, ...

The mapping is kept only in an internal file.

### Human-validation design

The same 60 cases are independently reviewed by Reviewer A and Reviewer B in different randomized orders.

The sample contains:

- 20 cases where both strong LLMs agree against SciGeoGuard-X;
- 10 cases where one strong LLM matches SciGeoGuard-X and the other disagrees;
- 10 cases where Qwen, Mistral, and SciGeoGuard-X all agree;
- 10 difficult missing-evidence / `UNPROVEN`-signal cases;
- 10 stratified random controls.

After independent review:

- A/B agreement is measured with Cohen's kappa;
- disagreements are exported to a **blind third-reviewer** sheet;
- final human labels are compared against SciGeoGuard-X, Qwen-7B, Mistral-7B, and the generated benchmark reference;
- bootstrap confidence intervals and paired disagreement tests are produced.

## 0. How to use this notebook

### First run
Run all cells once. It will create:

- `Reviewer_A_BLIND_60.csv`
- `Reviewer_B_BLIND_60.csv`
- `SciGeoGuard_X_v7_Reviewer_Package.zip`

Send the two reviewer sheets separately.

### Second run
After both reviewers finish, upload:

- `Reviewer_A_completed.csv`
- `Reviewer_B_completed.csv`

and run the notebook again.

If they disagree, the notebook creates:

- `Adjudicator_BLIND.csv`

Give that file to a third reviewer without showing A/B's answers.

### Final run
Upload:

- `Adjudicator_completed.csv`

and run again to produce final human-grounded metrics.

In [ ]:
SEED = 20261002
SAMPLE_N = 60

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import json, math, random, re, gzip, base64, shutil
import numpy as np
import pandas as pd

random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("/content") if Path("/content").exists() else Path("/mnt/data")
WORK = ROOT / "scigeoguard_v7"
WORK.mkdir(parents=True, exist_ok=True)

print("SciGeoGuard-X v7 human validation")
print("WORK =", WORK)

# Part I — Embedded v6 source data

The notebook embeds the actual completed v6 evaluation bundle, so the 60-case selection is reproducible without additional uploads.

In [ ]:
EVAL_GZIP_B64 = """..."""
BLIND_GZIP_B64 = """H4sIAEHwv2oC/+1d23LbSJJ9769Q6HVNT11w7TfvWOt1tO3uGHm7d6OjAwGREIU1CGAB0LY8Mf++VbiQIFEgCQkUAfJMtGZ6hCJFVmVm5eXkycT76nvfnCiZecmrqZt6jj979S1KvtwH0TfHDzMvzF65Sebfu9Msdf43jcJXaebFxb/+RF/dvvtv7vz7h/ef3jriJdHMmXlx9uAQSl9d//P6eh65wfX1z1fX17Wn19evxC8S7/+WfuLNnP9bumHmZ49yXbgMgo2Hy9DP6g9m3tRP/Sh0Mn/h1R8s3O9O4qVRsMzk44Xi3VLnzk08xxPf50E+vneD1Nt4Pg2i5cxZuOmXjefyvZfh1EsyV2zJxgfNvEUcJW7gZFHgJa5Y5Dwov8hXN1h6jlgxX33sf12/uv5TbFLoFl9FvluSiD8xsUmxR/Wtub72Au+rK79d8bDaGvH5il+kUzco32gaideFy2iZFo+mSVo8uPnt9t3PnBkar05hc8tssvGtao83vlacRFMvTf1w7ogP5QX1Z+kyEdLiOYto5pWH//b2Y/Hnvoo99MXHdGZutlyUH+ndR0aIVSxwp2K/Uj9THbH71fUD9y7wGk/Ex/nqhXL3i7f8exR7SehPl+nV25vyT9+54cxJomDjhWEkPojrpN5CbvQ0rT9rOXE3EKpRHejGC6K71Eu+ikcLP1T/3v2+8U5ZVhzLP//1r1dXG4LwzRWiMCl3UiUL+YJq7zelYdqjOFDy+ggCUT1QScMf724tzfGCwI/TyJ/1KRUPj7PEXQb+9Kr8JMMXi79KExHF1UEu7vzQc6qdK75D7CbuoniJeLn83/zQ7vITLGVr/Q6hfHFpWJuvLm19/tL85KOVjZEf5yfWavK5BpP/dJNPT2byKUw+TP4pTb6QBtuAoR+Iof+yYKWp53VT70XOIpBf2p/lBskhmr1l77eXwOi3Gf2bXyepu4gDL51wptkqbc/krSB1q1y4qfIrPRuhwq/Evxddd5fZQ5T4mZC4r95EPJ0tp9kYNT6NA6Wiyh2tXIXigIpFuYA4XjgrHjLCtAkxJ4R+JuTn/J9/y/+7XO2lmZNmQiXqyw3V8lz3tUL3//Pmw9tf/+vz5D/efHz/4X9yLXVm4kTCXDcXfrpws6nw/QjbsgUtK2ES2kzCzBenE6pv/s1nR3ICYQQGce0ffGmT+qU9rcIzvRmeJd4iyjzxZcP81Ag1lXHa5jIoapui5rs1kX9/wpQxW7Gd3neZO917awsnz00eOyorg6q+tKqK01nJnePG4q72Ziv5a8Ryt1QlGHfu9Is4+kyEccf05oYiGzWfRP9MFT7JBdj3bblgKrlIvPvAmypu+AuQC8YuUi627/3yCtq43Hbf/o1ofxHLi1Ne31GStqUL6CpdwAr3YeNCpGzrXcU7uoe4IQtPnEMZiMw9IR6pPy1eFy2zeJk5a591nV4w6q5KII5rKg43caYP8v4VbgrfclO2l8BFaXNR/oiSYPZ3uVUtLorcSiffy/0Gx828udCxVQ5q0+JowuScn5Nyc/vmar2JxzYof0rbLLeLix9N/OjixxA/pvixxI8sytr6K2HDyV/9W54iwdRVze/8wA+FDpTpBTeZe1lDc4z8Zd5Xf+ZJkY7undCTsIL7KFmUKcKfr7Jk6eUGwdwfuxAdsUs/sQs9ZuzSbhUoQpehhC6F3iFy2emhUkQuiFzUcmEgchlM5MJfMnCxlLWRdOqFwsUREhW4sRBB94srwxiyXSVVroPL0uayTB/8OFXZHn8h9xfVUVRHa3pVFcTqKnafCI0vYw1xmNzOldjek32gyD48P/tAT5Z9oMg+nHP2QYjQ9EeU9ph8ELuwZRDSQLzSEau9qZtm4iZXJSTzRVfVol6NQuVtWhOqf6aKPPlZWIv3H2/+8W5y7wlVSpTgBwlzER+rgXlcXBz6oSYQhhLks21IPr0RluTdbx+v8k0eYdSJfphLBUdvXxSxMCj+VOUBlqajuMvIX1uBYBWe1s3gbixscRuUnW1vb36/+fDrbx9vPn127pfy79QvEAXuLU+gHvs+kNmHc70Pcn3HddDpOmjBfJ5tMNjBGEhdZk1drsUVeZdrEkdBpdLbbl7rWoSAbUq8CpZRfx5P2maUgaAsQXvVRdtPIMib5mIZpstYyFHmzervPpcIfEJIA/veuhomo/3eT1fVxU1jUWmhUAPhxWaPx8v1wmQMoqB0akBKiemVtkBr7XbWOLqd0e28Cug/I6BHtzMILkbd91yW9+khPVQWcIj94BD5iXqoOHw9ABHHAzijBEBEABEVcsEvVS6GCURkWg9IxDhwwyqQ3YFDpMZughaqg6Dl+QQtpk7Az4K8VH8IRGpyq9BfczcGUWPAIAKDeAq1rX6LYtPGvTxd3lXtAS35Zd6t1GQ1S01pLNaJc83NTa2NYLvIpFwHg9BmEDZuadzkuMl33+SFcpV3uSdXrU9R55ZVKEzshYWe58JXWgBTL293e493zuGdP987102wJ0Kne9BpQtp1mlKr0GlGlK1/C7841/Xuigt7u4GouQgK3qbgy/BLGH0LJ34YL7MOlWIodqXY0f29PxXiPjKd/vOvnxhV6lgdhbnw3NBx5/PEm7fhsHcth97txW0eN2TGdXo2oXSlVp7TOPPD4+qFN/PdsBowwQ5svTDQeoHWiwG0XlALrRctrReM785wMx0ZbnD8IcN9thlupu3rqmfsVF31BF31uMs3BOLCuuqNk4HwDXTVX2JXPdPbhwfaaKdBOw00+QTtNJgihPaZ57fP/FKl74yWyWGrXVmVy5rcmFINc4vlBleV5T4H6y/UQu/f+j+4P9xkptL24oksNd65si92u2lajhQrR7MFQg1hAI6epRP7QunRS2fmbsgJZ4CcPB9ywiwLkBNATnqkpNVss7g7rT36S6C/z9dfqjMoMBS4V05pixUKvAfzySgUuIeOLEahwFDgnkZma8Kh6zAyW28dmc3JHkp5C8VutHOh2H3GJPKcduUOtMAdCO5AOASj4A58bWz8x+xEJciZMi8ePzym+dEW7SZf/TUP8XZyvG0p7EKbXfDDmfddZRY+vf39PTLiY7MME/La1LllmAbTTMuwdcpUhoK8tkzONMNitmmLf+heDHuayorXhkgdYD/KD0VLkV7DHDg/vK3bQFs32rqh4gNoAbU41dtbQDVqkUK59yNaKRCtQLQOA9GqfybGPkTr8bMDmAwF5Nu5YFj5AezAhIMdeATswO2pXpADgxx4ZNPoQQ4McmDIxbDJgSl7OXJgbjSTUPmZzRNfbmm6cLPpg4I5pbkInkpr4cmbjdHgXF9/YG/KSGjobkptk4d660TZg7r6+CJyYFyKHIR+MjA5aGYgxWkn+ZjQVeFpw7BLw9Iw7rn4OI3GGu+7fDfxguK5Yh/EisCf+pnjBv48XOTB0EbJ0VSWHDeRjOtSxDYXh3Id7oK2u2D64MfKQoS/kPuLKgSqEJ1AxbqtFVq8pymAoamnB0yxxXSGKiL0tz/9NQjhhQLbXZE/DMgfIH+g0DXkjxLow14a2KORZk5FvOsycKX/7rkK1a0/hrrurvjEUfA4r6zaC1d8KFT19HfvEdOhGu14CVOCSxiXMDR7A35rMWZqlm4apk1Namtq+K2p25xoBje5WG3r2otf0+zA0gdB6QOlD5Q+Rlv6YCh9oPShLH1ofD9Kj1Kg9PpB6bFjovQMm6PlcugovVLzANPbOatdB0wPMD3IxUXA9OaeEI+0GqawKzOlNaPVbfFTtBQ0JRSOitpRieIaFe5JTM7r58Qpn39FvNq/hak2syEBu0yN1FZ9z0QjBpInTDQ6jZ7e3L65Wm/iWY8dPCblU7cJR5oCZ1/u7w8viRRZ5tpTGINdswjR5X2qzMJqWnW1zSe8se/9ICg16hBtzz9MXhiyxX/kb3I19NNVrTd4LMCRm2psHuSFU3jhl+mFJ+435+0nOOLDccSt1llS6snAmCWFWVJg1DjyLKkp3DAMk3r+MKkviyo5au+v43LUccG2gjou2FZeilUDZVyUcVVyYaGMO6QyLn/JKq6uaA26X0r6uXrmWMSmpspZOToxKD1fYtA8aAUvaDdeUCEP+v5J92djkDpwREpdpgdQPBIEHQg6EHQg6EDQgaDjtHJhIOgYTNBhvBzDo852EwNpHMRAPQwb1TUQA4GcoKdho+aE2R2GjZqtw0b1AxrcNAMxygga3DBUEB1u59LJRBClIEpRyAWI6C82StF2RyncQJTy/CjFsG1EKYhSnj0Ekeo6bR+CyBkrOIl1XcmmVvvmTtlAoOhclfjI/E+6wVWloOeg5EIB9P6V/MH94SZKqqXiiexSuHPlAOvt6ebgVHtpVRf7wjk7qrL/+ddPutGOfGdAvj8d+c5Phnznx0G+fwbyHcj3AwxsLg220acsAP7+DPj7L1XsZB7AYqYhydtPkpcSME8jx4scL3K8yPEix4scbyf4u3V4QqjhsSAhhITQ2SaEiKUdPSFkN3tPlmG6jMVhSBbl2mnMJSaFkG0N3LEaoUN740maqcnPK1EWsiRM27Ya9usIQPGGcOE/jY1MGunQq4qoLXRk5RHv4SNbU58bik60Gh2eeKGQ2ng9bG7bFrSuhSVoswQrjseWHMJLEBcijXAJ/IXSWnhVm2iLwbA78RcaBzS7MR05RuQY4SWg2Q2E+EgxnlYuOFKMl8mwYbBt5vU0EC6OU1FnKNyU1aKj82vo58uv8f7jzT/egWCjI8GGphSITXtUvdGorqQKLGWcDCxlgCZ0bKQqG7dEdXfVDd9ubExh//nuRgJqoZHg+Y0EFjcpGgmQ0+6t3Zl3aHe2WtudDa2lvuxnzqqoWR/tzBsVLuVK2IQ2mzDz01qEuWkMNp8diQscVmAQGNiDEaxESeBs6AfN1WCYq3GZczUwjX1IQzUMYw+lEIGP/Xwfm5u2BR8bt+shzbpTL/Qqr0r8qyNBBIEbO/eJUNyylkte65ZlFAps7h5P2aAfxnhKjKfEeMoxjafUusE7rAPgHeAyBk8YbnP0kL0cay0FwAMAD7VcAOAxHIAHZS9HFGbYe4jCKHIPz889ME5M5B6Qe+gv96CZhp4rsKloPBFyuQzcxJF2Q5F9qD+G+u4OMOIoeJxXVu5oAYaGAGOwKnvEy9ek+8CVDZ7OFwNXUoArAa7cFgh+1uDKs2OiA7hy2OBKk3UgdKXg7wB/x8UQuhYycEz+DpMrOvbXyeV2xWsuQhDVpna3bLLCGqnZOgCXOm+41J6KTR8ZWHmTanvb1Cy0qSGSGkabmqFE559RJGWfLJKyEUldZCSl77P/lMD+w/6Pp035+vrTm9s3V+9++3iVb/IIYRZnextgwsPAbwNj322gnY60guI2wG1QFwh27tEAJvzA/r+s/d/Tj8PRj9NDPw5YV9GPczqCZtapIce09nmE5GT5AUbgEcIj3BQIYoPGbEw0ZvAIB+4R2rs9Qo3CI0SH9ik8wuPb8VH6gNPlXcVO2+IAGp0cQIvsMQAcBgAhIULC0ZoD3s0cUCXydrEMMr/o3cxPrc5tuD2xp3UtLEUrGvCU/fiv0ZGPjvxRSYaNnvzDW+pr8+BbWuql2We7u92ZiW73HrrdGbrd0TrbK9OeOOtCgRVNG2ksNkqcRv7XxDG6X9ycdM/e0mXlOih0m0JvKCkUGYq8W5EL5SpV2ZOrarGZbbNCYWIvLOKzXPjK3J3GtUK5taZyi7cRyzw3dNzUcafTpQi7qhmq2ymb1rVQ8p31OZWKx4n43nEuuuLLyZ1tVuj+9tCxLrMrfUP2zFQWlwD0/Nh6vtp9dz5PvPnq1A9P1Ej1K6Uivxuc2bK0BPVTFEoVuNuqKE+jrrOFTdD3zJ8BN3YPHjvVbQMXPQzAcy963Sry8OqLnmpmGYYbzYv+fimrvvWErcKFzymXgNB+us7n6AvAcTrAccwJpwe165yNDegAzZC6bCorKfHDY5ofQ6H/X/2octvpttvethQ3d5sW+8K2flep76e3v78He8nYLuwJec0NyyDM1HRqmpxTW1MpL3ltUmLrFrGp8Hstzvle7ro0lXOkNoTqAH++/Fi0FOo1jMo6gNe+0VoFXvsn8trzY/Lat8fiHHo9FJKUAq8AVvud4+kJauhgtVfLBVjth8Nqz1+O1N6yD8wvaMgvIL9w8gZwekC7z8XmF2wFv/0yTJexONvMm9UPdy5nYTf5GnesRhjSrsjpypNUEzcKCYzF53g8nlehIxAZgvNwxCZeyop64B7Y9oor8iebKkhb1z0D4oVCauN1rnF7wnXrWliCNkuwaoRBe9d4LMEomzykufAqt7utzUPv0uZhqwnWV4JUQAErzCGh266DeiFsRavX4E2jZKbGDMoWGSFLPlAEQBGsDqjUdRkWOl44qwdGxoRSJTNWJmyEkwr7kDVaJraXF0aAd/IZiAGfoT+fAR2h8BlOPbubdfMZtANqnJjd3VONk55odjesA2Z3j6qapaPKiSqnUi4YQ5VzMFVO4+WqnLbesTLSQF6iMjLSygggWudeGTG7FkaMDqMkCUZJYpQkLEJPgyTtrh0QxEQHBDogoHQbHRCEawaxKLcNRk2dGLa6A8JiuqVzWzMNapuWrb90B4RtdaooNjgKUFFERRE362kqiuaE8H4qigp88dvbj5J25O3njwoHe/0Qmt6m6SU3PEYFQeGfwvNezIGmhBweBzfQgYiDEQef520t9oVZ9NiRMCW0m3dM4B3DO4Z3PBTv2OzFO6aEHdh+R9B+h/a7IbTfGWi/a2m/o4TvHpzCLAxOweQkTE4a1/TMbsOTKNEOvNEt3Oi40Qdwo1OKG731Rtf3zcKlxqlm4UqYK2bhQpc3BMLCLNwxzcK9xSzcQc/CpUSNFczHJ6TLhfPNFxfjMltTtxOiq0YtqNYi2msz/8VEhcl6lMLgBy5QZG6HP3BBdiTP1+LSMnWBdpu6QInZeT6mhfmYmI+J+ZiYj/kykqEps/aYj/m0+ZhGafYV/N4zL84eFKMxa09h0feA2ib2yYI8G0HeEAz4N+GhJZNyJ1WykC+o9n5TGqYjzv9cX//x7tbSHC8I/DiN/FmfUvHwOEvcZeBPr8pPMkrrfeeHnlPt3AHmu9G2HMoXl4Z1d/xfWP7tRmJK7Hazr8HsP93s05OZfQqzP3azP2qrfwRhgLV/hrX/pbL1VA2Q35y+vRqi3YDIK9fhImi7CKYPwutR6b2/kPuLsZoA6Nb0qjL4dRW7T4Q6l4AccZiWWaox3TMX18Rc3OfPxTUswjAXFwrcnwIzapUKrKam3jyRen1l+yZuWQm9HmiD6Vn44OdST8nDrkuKx46YhTunSkr36Cy35VxdLS8bjdcbpuh5ai6CBW+z4MvwSxh9Cyd+GC+zDnYcDtmYcJJlLzFV9ByksTfN8k0XH14x6mvjOfSovW14NkYS3evrD+xNeQkPPaKpbfJQfaAoe1D7QJCDPuUg/eYn1BiYKDTj29Jw1oj1NhwfaVsaOMRcgpxGztn7Lt9NvKB4vt6K0C/lTawI/KmfOW7gz8NFzvJf7zejegc2UwssLmBxuRQWF0qN43texu6Wb81EyzdavtHyPa6Wb61byzc1WzFBnAETBCgoMEHABAETdCYIUGrthhNogBP0ACegBjcAJ0ClqS/CPtoTYR+1u06S0jFJ6jwmSekwB2c+SYrqpNMkKcpI195vStD7jd5v9H6j9/tlJIN/Jhy93331flNW2n3a9AKFzVoGrqzxea6CBqj+GKZ997jzOAoe51F45HHnGgaGDlZbjzgLmDLWmq6nSNejhRfpeqTrT2jDkafvJ0/PeKfuXYLuXXTvwm8bTPOfbpdqrDW9tcRbRFkRyuW4Rs1Wum2by6C7u4Mu+fdbkFY9BV2Gzfm5waxGq7vicFZi57ix0FtvtoY1befVTplxHV5WTTKtI9966nzrIOWCMWRba3zr9att30zarQDgoHQtXUUQTJGuZdrWuz4lrTT3hHik/vSAxJLezg1nI7GExNKpRl0jsQQcKPJLvXLDMaPb+GYD45sxvhkx6iDQoMaE8n7QoEw9+yN+eEzz48yF2vnqR0HbHOe2pbAEbZag1na9aQc+vf39Pdpux2YNJuS1pbIG9DXfCxJIU3mHb0jNAbFd+XdpKbU1ciLWPtNBowjg0MiHAA4BHAK48wjg7I6kogSkoiAVBakoSEWfKgu2AUnok0qUkyYIv0b04shxn0m8Cr0bCK/WtTDhbSZ8xV4ESp7xKO0oeXqEJE1/ROnOdk2jG00Pp10AoUQDIBSAUKTohgII1YhWqjHbDwglJgCh/QBC6WkAoRS6C0DoeIB/BIBQAELVcgFA6IAAocaL4kE574YR0oERAkYIbstAMEIK0M+TMEJc200R3uAcAEU4KMJBET7s1GNHhnCud2WN1MAaeR6skShXnD1rZOn2HU4byY1OlQgDlQhUImADhlKJ4LZRqrF5QCWCoRIxAmoKwA1GUIkob1CUIlCKQCmis1wYlyoXZ8xNcSjlKbea+YfyfH54SaQYD117Cu+kPb+ghrPL34tP00Qwg+SyR2MiW/nStft8Upty7wdBqVGH6HD+YVb/L1dBP121DQePxbSorRyivbuOwFFHQB3hROp8c/vmar2JADQ/saqgd6sqaIoGiESIn7Pw3NBxU8edTpeL5Yp7YLum0LoWlqLNUrz/ePOPdyoLESfie8f5hSa+nNzZ5uX/t4eO13+7sRCSTXbbC/JaR17i2Nf+avfd+Tzx5qtTP9wsyMaj+VpcCtvizJaJ2zhKoVmBu62P8kjqiltaBro/M0mRmRwDRnqXDUBmEhhpkOYiMTluuTCRmDw3jPSheUmN7c5pNCDRyGkgp4GcxqjgUR2hkhpvJjXybZ8n/qzOx7NdsWgugnFo75SYjdEHub7+wN6UgfLQI5faJg/VEY2yBzU6FnLQpxyEfjIwOWhC48RpJzkNzopbdeMWkIalcRPk4uM0mNa87/LdxAuK54p9ECsCf+pnjhv483CR50c2rgCtlYaTc9BwgoYTNJyg4QQN55hpOBerDED7zBxNh63HzBwMY4ath60/D1tvdCLsbLTOgLDzGYSd/GRJYI469QXkgos2uh15YLtjHtg8hJ99GX4Jo2+hIh2sXghLAXZ2sLODnR2ycHx+ds3qSHpCbJCenAfpCVy+cyc9MTpSnmh2Ny5EBi5EcCHCBgyCC9HsjQtRJ1150BhcgvNwCXSYg7PnQevoE+i0aQ1WUBS5pwo/YOM5NB4APwD8APADwG9cAD+dtYI+GAHo4+mgD34y0AcHwG/soI8vo07h//Hu1tIcLwj8OI38GdAfA0F/1Lr9dN6J9piC9hi0xwj5h0J7TEol1vYH7QRBO4J2BO0I2octB96s8h8QuB8YuOv7eaUYeKV64pXip5m9C/AGGO9HRSBEQCwFYimlXGD47oCIpSh70em7urHNLZUG/sxz7qPEm7pp5hCuKdil8kVX1aJe/ZRKNC0JkKEK0TwfitzJvSeUNfHAjr/Li9kQCGI3BWLbIn16c/vm6t1vH69qPMSjuqLOlokC3ckd76RYGBR/qspMlqajaIUjf21dGtVdVjeDu6sR5XVg7r0OyKmuA0mNiusA18FaIDSlQLz0+OYjXgDGyS4AAxfAZV4A1m6uWWqCa/b5XLP0ZDQD46XDP74hP192WaMbq4DerQ2N6mhDQxsaqhmDaEMzJkzrpw3NIO34Ywr8MQhG4dWDdK4zg4htAHQ8GNDxytTTVlNPNZh6mHq0mqDVBLZ+xLb+l5WpZypkurBGq3B93VvSbCtXrMMl0HYJbDSPbIX18naQOoYGE8T2G8pVmnxPrqqdosXMQmFiEe/nSbxc+EosqqGV6XtD3TyWz7BOlwvnmy/2c5mtR+c2lbx1LRS9TdGLsdaT9YBiTL2Gxj9/6nW6XPQ679rYmBjkRc4ikHvgz0pTwLeZJbeXwAK0WYCbXyflTT4xuanjyocB6K+nVNdL/dWVl7tQ88wvMMf55tbGPzaSN61rodptqn3SPpLXA+wYkLBsdJKcupNkkJLB0EvSqRWkludRtIIwrbT7Rlc+UAI+0PPgA6UEntyZE4LqpSk/nBHUMPf3l2sU/eX99JfTY/aXj6JZ9OIbzAtkJPrLERWgv7yzXPBLlYth9pfzF20vN6xWYBExASzC4GIAiwAswuTi85hcbNi7ewc1Db2D6B08TUR6c/vmar2JmE38lNnErFsXoUl2ww0axUnADZ4AN9A50wA3QJL6uQhDw9KMdoQhtQy71GqqBCHUvrrEFBRJaLqdhJZOd/433eCq0tBz0HKhAXr/Wv7g/nATJdl18UTWn+5cWWTYLkGJnREbEsqtEaqfXhpFzAl0XewL17Wjavuffwnt2zETyEA+BY1ayKcgn4JGrfOYBGTy3fmUxuwf5FOekE9hJ8unMORTzjmfIpE/XsWG2Q8xk6ntR/5wHciffpA/7DSTJRgiLEyWwGQJIH8wWQLIn9Eif0y92bQgbt5l4MqZW54rwpftdtP6Yzgoux2UOAoe51Ve6qWhyZh9NQCj8hT1jUVYW7Uj7FReY3fplqB020Pp1tJ1jtItVLkf4ldtQnkH4le9nfjVNJUlXiGkstLoCIPi5hsZRGmqyEIq18EgtJM/b8UH6l7Di2srvqDkQ6Uxgfsokwd5LPn3D4U6blqKWRLFDUU8xAtYyeQs+hammfAdFltdhabVmV2Cg10C7BLoIwO7xItJBkc+qS92CV6affWsj00+ohUlaAPRp1wHc99m7qcPfqwkBPUXcn8R8CHg60QNxqlRqLFF9g1uJAYGN2Jw4zAGNxrKRMXZzvHlJwOXcoBLL3GM4/8DvXIy2RB6AwA="""
V6_METRICS_GZIP_B64 = """H4sIAEHwv2oC/3WMwWrDMAyG73uWUOzYie3rBu2ph62X3YRwlBBo7SI7HX372Wt62FgEEtKvjy/dU6ZLg94vjP4OtwQTBWLMNADTSEzBU3NBzxFGufXfn94+tn4jnhPVe8mY5xigAlvwFbnAkeFGPMw+g49lw4leTpljmOD9iwKY10bs9J8qUeuM0653qpPC2s6UqHe9bssq11kpK9uukTvxdB7nlBnPD62y6lf1NRNKFqNwxkrttFT/i6VwyqxmPx8oHhbkAT5r8tNi7Up8A0C7cbN6AQAA"""

def decode_csv(payload):
    from io import BytesIO
    raw = gzip.decompress(base64.b64decode(payload.encode()))
    return pd.read_csv(BytesIO(raw))

v6 = decode_csv(EVAL_GZIP_B64)
blind180 = decode_csv(BLIND_GZIP_B64)
v6_metrics = decode_csv(V6_METRICS_GZIP_B64)

print("v6 evaluation:", v6.shape)
print("blind cases:", blind180.shape)
display(v6_metrics)

# Part II — Parse split and family internally

These fields are used **only for stratified sampling** and are never shown to reviewers.

In [ ]:
def parse_split_family(case_id):
    m = re.match(
        r"^SGX3_(BLIND|DEVELOPMENT|HELDOUT-FAMILY)_(.+)_(\d+)$",
        str(case_id)
    )
    if not m:
        return "UNKNOWN", "UNKNOWN"
    return m.group(1), m.group(2)

parsed = v6.case_id.apply(
    lambda x: pd.Series(parse_split_family(x))
)

parsed.columns = ["split","family"]
v6 = pd.concat([v6, parsed], axis=1)

v6["both_llm_agree_against_sgx"] = (
    v6.A_verdict.eq(v6.B_verdict) &
    v6.A_verdict.ne(v6.frozen_prediction)
)

v6["one_llm_matches_sgx"] = (
    v6.A_verdict.eq(v6.frozen_prediction) ^
    v6.B_verdict.eq(v6.frozen_prediction)
)

v6["all_three_agree"] = (
    v6.A_verdict.eq(v6.B_verdict) &
    v6.A_verdict.eq(v6.frozen_prediction)
)

v6["unproven_signal"] = (
    v6.generated_expected.eq("UNPROVEN") |
    v6.A_verdict.eq("UNPROVEN") |
    v6.B_verdict.eq("UNPROVEN") |
    v6.frozen_prediction.eq("UNPROVEN")
)

print(v6.split.value_counts())
print("families:", v6.family.nunique())

# Part III — Diversity-aware deterministic sampler

The sampler favors underrepresented splits, families, and generated verdict classes within each selection stratum.

This is an **internal sampling mechanism**, not information provided to human reviewers.

In [ ]:
def diverse_sample(df, n, seed):
    if n <= 0 or len(df) == 0:
        return df.iloc[0:0].copy()

    if len(df) <= n:
        return df.copy()

    rng = np.random.default_rng(seed)

    temp = (
        df.copy()
        .reset_index()
        .rename(columns={"index":"orig_index"})
    )

    temp["_rand"] = rng.random(len(temp))

    split_freq = temp["split"].value_counts().to_dict()
    family_freq = temp["family"].value_counts().to_dict()
    label_freq = temp["generated_expected"].value_counts().to_dict()

    split_counts = Counter()
    family_counts = Counter()
    label_counts = Counter()

    remaining = temp.copy()
    chosen = []

    for _ in range(n):
        best_idx = None
        best_score = None

        for idx, row in remaining.iterrows():
            split_bonus = 1 / (1 + split_counts[row["split"]])
            family_bonus = 1 / (1 + family_counts[row["family"]])
            label_bonus = 1 / (1 + label_counts[row["generated_expected"]])

            rarity = (
                1 / max(1, split_freq.get(row["split"],1)) +
                1 / max(1, family_freq.get(row["family"],1)) +
                0.5 / max(1, label_freq.get(row["generated_expected"],1))
            )

            score = (
                2.0 * split_bonus +
                3.0 * family_bonus +
                2.0 * label_bonus +
                0.1 * rarity +
                0.001 * row["_rand"]
            )

            if best_score is None or score > best_score:
                best_score = score
                best_idx = idx

        row = remaining.loc[best_idx]
        chosen.append(int(row["orig_index"]))

        split_counts[row["split"]] += 1
        family_counts[row["family"]] += 1
        label_counts[row["generated_expected"]] += 1

        remaining = remaining.drop(best_idx)

    return df.loc[chosen].copy()

# Part IV — Construct the exact 60-case sample

Selection strata are mutually exclusive because each selected case is removed before the next stratum is sampled.

In [ ]:
used = set()
parts = []

def select_stratum(name, pool, n, seed):
    global used, parts

    available = pool[
        ~pool.index.isin(used)
    ].copy()

    selected = diverse_sample(
        available,
        n=n,
        seed=seed
    )

    if len(selected) != n:
        raise RuntimeError(
            f"{name}: requested {n}, only selected {len(selected)}"
        )

    selected["selection_stratum"] = name

    used.update(selected.index.tolist())
    parts.append(selected)

# 1) Both 7B LLMs agree against SciGeoGuard-X.
select_stratum(
    "LLM_CONSENSUS_VS_SGX",
    v6[v6.both_llm_agree_against_sgx],
    20,
    SEED+10
)

# 2) One LLM supports SciGeoGuard-X, the other does not.
select_stratum(
    "ONE_LLM_MATCHES_SGX",
    v6[v6.one_llm_matches_sgx],
    10,
    SEED+20
)

# 3) All systems agree: important controls.
select_stratum(
    "ALL_THREE_AGREE_CONTROL",
    v6[v6.all_three_agree],
    10,
    SEED+30
)

# 4) Missing-evidence / UNPROVEN signal cases.
# Ensure any remaining generated UNPROVEN cases are included first.
remaining_unproven = v6[
    (~v6.index.isin(used)) &
    v6.generated_expected.eq("UNPROVEN")
].copy()

unproven_selected = remaining_unproven.copy()

if len(unproven_selected) > 10:
    unproven_selected = diverse_sample(
        unproven_selected,
        10,
        SEED+40
    )

used.update(unproven_selected.index.tolist())

need = 10 - len(unproven_selected)

if need > 0:
    pool = v6[
        (~v6.index.isin(used)) &
        v6.unproven_signal
    ]

    extra = diverse_sample(
        pool,
        need,
        SEED+41
    )

    used.update(extra.index.tolist())

    unproven_selected = pd.concat(
        [unproven_selected, extra],
        axis=0
    )

unproven_selected["selection_stratum"] = "UNPROVEN_EVIDENCE_STRESS"
parts.append(unproven_selected)

# 5) Remaining diversity controls.
remaining = v6[
    ~v6.index.isin(used)
]

random_controls = diverse_sample(
    remaining,
    10,
    SEED+50
)

random_controls["selection_stratum"] = "STRATIFIED_RANDOM_CONTROL"
parts.append(random_controls)
used.update(random_controls.index.tolist())

sample60 = pd.concat(parts, axis=0).copy()

if len(sample60) != 60:
    raise RuntimeError(
        f"Expected 60 cases, got {len(sample60)}"
    )

if sample60.case_id.nunique() != 60:
    raise RuntimeError("Duplicate case IDs in final sample.")

print(sample60.selection_stratum.value_counts())
print("\\nGenerated-reference distribution:")
print(sample60.generated_expected.value_counts())
print("\\nSplit distribution:")
print(sample60.split.value_counts())
print("\\nDistinct families:", sample60.family.nunique())

# Part V — Anonymize case IDs

Reviewers must not see benchmark family names embedded in original case IDs.

Neutral IDs are assigned after a deterministic shuffle.

In [ ]:
sample60 = (
    sample60
    .sample(frac=1, random_state=SEED+100)
    .reset_index(drop=True)
)

sample60["human_case_id"] = [
    f"HV60_{i:03d}"
    for i in range(1, len(sample60)+1)
]

internal_manifest = sample60[[
    "human_case_id",
    "case_id",
    "split",
    "family",
    "selection_stratum",
    "generated_expected",
    "frozen_prediction",
    "A_verdict",
    "B_verdict",
    "A_confidence",
    "B_confidence",
    "workflow_intent",
    "artifacts_json",
    "steps_json"
]].copy()

display(
    internal_manifest[[
        "human_case_id",
        "split",
        "selection_stratum",
        "generated_expected"
    ]].head()
)

# Part VI — Create blind reviewer sheets

Reviewer A and Reviewer B receive the same 60 workflows in different randomized orders.

Only neutral IDs and scientific workflow content are visible.

In [ ]:
REVIEW_FIELDS = [
    "reviewer_verdict",
    "primary_error_class",
    "severity",
    "confidence_1_to_5",
    "reason",
    "reviewer_id"
]

def make_blind_sheet(sample, seed):
    d = sample[[
        "human_case_id",
        "workflow_intent",
        "artifacts_json",
        "steps_json"
    ]].copy()

    d = d.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    d.insert(
        0,
        "review_order",
        np.arange(1, len(d)+1)
    )

    for col in REVIEW_FIELDS:
        d[col] = ""

    return d

reviewer_A = make_blind_sheet(
    sample60,
    SEED+201
)

reviewer_B = make_blind_sheet(
    sample60,
    SEED+202
)

A_path = WORK/"Reviewer_A_BLIND_60.csv"
B_path = WORK/"Reviewer_B_BLIND_60.csv"

reviewer_A.to_csv(A_path,index=False)
reviewer_B.to_csv(B_path,index=False)

print(A_path)
print(B_path)

# Part VII — Reviewer rubric

Both reviewers should receive exactly the same rubric.

### PROVEN
The supplied evidence is sufficient for the declared scientific workflow and no scientific contradiction is present.

### REFUTED
At least one actual scientific operation, transformation, physical assumption, geospatial relation, temporal relation, or evaluation design contradicts the declared scientific goal.

### UNPROVEN
No definite contradiction is established, but evidence genuinely required by the workflow is missing.

### Important
Optional intent fields set to `null` mean **no additional constraint was declared**. Do not mark a case `UNPROVEN` solely because an optional intent field is null.

The reviewer should judge the workflow itself—not guess what any software system would answer.

In [ ]:
REVIEW_INSTRUCTIONS = """
SciGeoGuard-X Human Validation — Reviewer Instructions
======================================================

Work independently.

For each case choose exactly one verdict:

PROVEN
The supplied evidence is sufficient for the declared scientific workflow and no scientific contradiction is present.

REFUTED
At least one actual scientific operation, transformation, physical assumption, geospatial relation,
temporal relation, or evaluation design contradicts the declared scientific goal.

UNPROVEN
No definite contradiction is established, but evidence genuinely required by the workflow is missing.

IMPORTANT NULL SEMANTICS
Optional intent fields set to null mean that no additional constraint of that kind was declared.
Do not label a case UNPROVEN merely because an optional intent field is null.

Fill:
- reviewer_verdict
- primary_error_class
- severity: LOW / MEDIUM / HIGH / CRITICAL
- confidence_1_to_5
- reason
- reviewer_id

Do not discuss answers with the other reviewer until both files are submitted.
Do not attempt to infer the software system's expected answer.
""".strip()

instructions_path = WORK/"REVIEW_INSTRUCTIONS.txt"
instructions_path.write_text(
    REVIEW_INSTRUCTIONS,
    encoding="utf-8"
)

print(REVIEW_INSTRUCTIONS)

# Part VIII — Leakage checks

The notebook verifies that reviewer-facing files contain no machine answer, benchmark label, family name column, or original case ID column.

In [ ]:
FORBIDDEN_REVIEW_COLUMNS = {
    "case_id",
    "split",
    "family",
    "selection_stratum",
    "generated_expected",
    "frozen_prediction",
    "A_verdict",
    "B_verdict",
    "A_obligations_json",
    "B_obligations_json"
}

checks = {
    "60_unique_cases":
        sample60.human_case_id.nunique() == 60,

    "reviewer_A_has_60":
        len(reviewer_A) == 60,

    "reviewer_B_has_60":
        len(reviewer_B) == 60,

    "same_case_set_A_B":
        set(reviewer_A.human_case_id) ==
        set(reviewer_B.human_case_id),

    "different_order_A_B":
        reviewer_A.human_case_id.tolist() !=
        reviewer_B.human_case_id.tolist(),

    "A_no_forbidden_columns":
        FORBIDDEN_REVIEW_COLUMNS.isdisjoint(
            reviewer_A.columns
        ),

    "B_no_forbidden_columns":
        FORBIDDEN_REVIEW_COLUMNS.isdisjoint(
            reviewer_B.columns
        ),

    "neutral_ids_only":
        reviewer_A.human_case_id.str.match(
            r"^HV60_\d{3}$"
        ).all()
}

leakage_df = pd.DataFrame([
    {"check":k,"passed":bool(v)}
    for k,v in checks.items()
])

display(leakage_df)

assert leakage_df.passed.all()

# Part IX — Export reviewer-only package

This ZIP is safe to send to reviewers. It deliberately excludes the internal mapping and machine verdicts.

In [ ]:
reviewer_dir = WORK/"reviewer_package"

if reviewer_dir.exists():
    shutil.rmtree(reviewer_dir)

reviewer_dir.mkdir()

for p in [
    A_path,
    B_path,
    instructions_path
]:
    shutil.copy2(
        p,
        reviewer_dir/p.name
    )

reviewer_zip = shutil.make_archive(
    str(ROOT/"SciGeoGuard_X_v7_Reviewer_Package"),
    "zip",
    root_dir=reviewer_dir
)

print("Reviewer package:", reviewer_zip)

# Part X — Import completed reviewers

After independent review, upload:

- `Reviewer_A_completed.csv`
- `Reviewer_B_completed.csv`

The notebook checks verdict validity and case completeness.

In [ ]:
VALID_LABELS = {"PROVEN","REFUTED","UNPROVEN"}

A_COMPLETED = ROOT/"Reviewer_A_completed.csv"
B_COMPLETED = ROOT/"Reviewer_B_completed.csv"

def load_completed_review(path):
    if not path.exists():
        return None

    d = pd.read_csv(path,dtype=str).fillna("")

    required = {
        "human_case_id",
        "reviewer_verdict"
    }

    if not required.issubset(d.columns):
        raise ValueError(
            f"{path.name} missing {sorted(required-set(d.columns))}"
        )

    d["reviewer_verdict"] = (
        d["reviewer_verdict"]
        .str.strip()
        .str.upper()
    )

    bad = (
        set(d.reviewer_verdict) -
        VALID_LABELS -
        {""}
    )

    if bad:
        raise ValueError(
            f"{path.name}: invalid verdicts {sorted(bad)}"
        )

    if d.human_case_id.duplicated().any():
        raise ValueError(
            f"{path.name}: duplicate human_case_id"
        )

    expected = set(sample60.human_case_id)
    supplied = set(d.human_case_id)

    if supplied != expected:
        missing = expected - supplied
        extra = supplied - expected

        raise ValueError(
            f"{path.name}: case set mismatch. "
            f"missing={sorted(missing)}, extra={sorted(extra)}"
        )

    return d

review_A_done = load_completed_review(A_COMPLETED)
review_B_done = load_completed_review(B_COMPLETED)

print("Reviewer A completed:", review_A_done is not None)
print("Reviewer B completed:", review_B_done is not None)

# Part XI — Inter-rater agreement

When both reviewer files are present, compute:

- raw agreement;
- Cohen's kappa;
- per-label agreement;
- confusion table.

In [ ]:
def cohens_kappa(a,b,labels=("PROVEN","REFUTED","UNPROVEN")):
    a = pd.Series(a)
    b = pd.Series(b)

    ok = a.isin(labels) & b.isin(labels)
    a = a[ok].reset_index(drop=True)
    b = b[ok].reset_index(drop=True)

    if len(a) == 0:
        return np.nan

    po = float((a==b).mean())
    pa = a.value_counts(normalize=True)
    pb = b.value_counts(normalize=True)

    pe = sum(
        float(pa.get(x,0))*float(pb.get(x,0))
        for x in labels
    )

    return (
        np.nan
        if math.isclose(1-pe,0)
        else (po-pe)/(1-pe)
    )

review_pair = pd.DataFrame()
agreement_summary = pd.DataFrame()
reviewer_confusion = pd.DataFrame()

if review_A_done is not None and review_B_done is not None:
    review_pair = (
        review_A_done[[
            "human_case_id",
            "reviewer_verdict",
            "reason",
            "confidence_1_to_5"
        ]]
        .rename(columns={
            "reviewer_verdict":"A_human_verdict",
            "reason":"A_human_reason",
            "confidence_1_to_5":"A_human_confidence"
        })
        .merge(
            review_B_done[[
                "human_case_id",
                "reviewer_verdict",
                "reason",
                "confidence_1_to_5"
            ]].rename(columns={
                "reviewer_verdict":"B_human_verdict",
                "reason":"B_human_reason",
                "confidence_1_to_5":"B_human_confidence"
            }),
            on="human_case_id",
            how="inner"
        )
    )

    complete = review_pair[
        review_pair.A_human_verdict.isin(VALID_LABELS) &
        review_pair.B_human_verdict.isin(VALID_LABELS)
    ].copy()

    raw = float(
        (complete.A_human_verdict ==
         complete.B_human_verdict).mean()
    )

    kappa = cohens_kappa(
        complete.A_human_verdict,
        complete.B_human_verdict
    )

    agreement_summary = pd.DataFrame([{
        "n_double_reviewed":len(complete),
        "raw_agreement":raw,
        "cohens_kappa":kappa,
        "n_disagreements":int(
            (
                complete.A_human_verdict !=
                complete.B_human_verdict
            ).sum()
        )
    }])

    reviewer_confusion = pd.crosstab(
        complete.A_human_verdict,
        complete.B_human_verdict
    )

    display(agreement_summary)
    display(reviewer_confusion)
else:
    print("Agreement analysis awaits both completed reviewer files.")

# Part XII — Blind third-reviewer adjudication

Reviewer disagreements are sent to a third reviewer using neutral IDs and workflow content only.

The third reviewer does **not** see Reviewer A/B answers.

In [ ]:
adjudicator_blind = pd.DataFrame()

if not review_pair.empty:
    disagree_ids = set(
        review_pair.loc[
            review_pair.A_human_verdict !=
            review_pair.B_human_verdict,
            "human_case_id"
        ]
    )

    adjudicator_blind = sample60[
        sample60.human_case_id.isin(disagree_ids)
    ][[
        "human_case_id",
        "workflow_intent",
        "artifacts_json",
        "steps_json"
    ]].copy()

    adjudicator_blind = adjudicator_blind.sample(
        frac=1,
        random_state=SEED+303
    ).reset_index(drop=True)

    adjudicator_blind.insert(
        0,
        "review_order",
        np.arange(1,len(adjudicator_blind)+1)
    )

    for c in REVIEW_FIELDS:
        adjudicator_blind[c] = ""

adjudicator_path = WORK/"Adjudicator_BLIND.csv"
adjudicator_blind.to_csv(
    adjudicator_path,
    index=False
)

print("Disagreement cases requiring blind adjudication:",
      len(adjudicator_blind))
print(adjudicator_path)

# Part XIII — Import adjudicator results

Upload `Adjudicator_completed.csv` after the blind third review.

For A/B disagreements, the third review acts as the deciding human vote.

In [ ]:
C_COMPLETED = ROOT/"Adjudicator_completed.csv"

def load_adjudicator(path):
    if not path.exists():
        return None

    d = pd.read_csv(path,dtype=str).fillna("")

    if "human_case_id" not in d or "reviewer_verdict" not in d:
        raise ValueError(
            "Adjudicator file must contain human_case_id and reviewer_verdict."
        )

    d["reviewer_verdict"] = (
        d["reviewer_verdict"]
        .str.strip()
        .str.upper()
    )

    bad = set(
        d.loc[
            d.reviewer_verdict.ne(""),
            "reviewer_verdict"
        ]
    ) - VALID_LABELS

    if bad:
        raise ValueError(
            f"Invalid adjudicator labels: {sorted(bad)}"
        )

    return d

review_C_done = load_adjudicator(C_COMPLETED)

print("Adjudicator completed:", review_C_done is not None)

# Part XIV — Build final human labels

Rules:

1. If A and B agree → their shared verdict is final.
2. If A and B disagree and C is present → C's blind verdict is final.
3. Otherwise the case remains unresolved and is excluded from final human-grounded accuracy metrics.

In [ ]:
human_final = pd.DataFrame()

if not review_pair.empty:
    human_final = review_pair[[
        "human_case_id",
        "A_human_verdict",
        "B_human_verdict"
    ]].copy()

    if review_C_done is not None:
        c_small = (
            review_C_done[[
                "human_case_id",
                "reviewer_verdict"
            ]]
            .rename(columns={
                "reviewer_verdict":"C_human_verdict"
            })
        )

        human_final = human_final.merge(
            c_small,
            on="human_case_id",
            how="left"
        )
    else:
        human_final["C_human_verdict"] = ""

    def resolve(r):
        a = r.A_human_verdict
        b = r.B_human_verdict
        c = r.C_human_verdict

        if a in VALID_LABELS and a == b:
            return a, "A_B_AGREEMENT"

        if (
            a in VALID_LABELS and
            b in VALID_LABELS and
            a != b and
            c in VALID_LABELS
        ):
            return c, "BLIND_THIRD_REVIEW"

        return "UNRESOLVED", "NEEDS_ADJUDICATION"

    resolved = human_final.apply(
        resolve,
        axis=1
    )

    human_final["human_final_verdict"] = [
        x[0] for x in resolved
    ]

    human_final["human_resolution_rule"] = [
        x[1] for x in resolved
    ]

    print(
        human_final.human_final_verdict.value_counts()
    )
else:
    print("Final human labels await completed Reviewer A/B files.")

# Part XV — Compare systems against final human labels

This is the first stage where system accuracy is evaluated against independent human judgments rather than the generated benchmark reference.

In [ ]:
LABELS = ["PROVEN","REFUTED","UNPROVEN"]

def exact_accuracy(truth,pred):
    truth = pd.Series(truth)
    pred = pd.Series(pred)

    ok = (
        truth.isin(LABELS) &
        pred.isin(LABELS)
    )

    if not ok.any():
        return np.nan

    return float(
        (truth[ok].values == pred[ok].values).mean()
    )

def macro_f1(truth,pred):
    truth = pd.Series(truth)
    pred = pd.Series(pred)

    ok = truth.isin(LABELS) & pred.isin(LABELS)
    truth = truth[ok]
    pred = pred[ok]

    if len(truth)==0:
        return np.nan

    f1s = []

    for lab in LABELS:
        tp = ((truth==lab)&(pred==lab)).sum()
        fp = ((truth!=lab)&(pred==lab)).sum()
        fn = ((truth==lab)&(pred!=lab)).sum()

        precision = tp/(tp+fp) if tp+fp else 0.0
        recall = tp/(tp+fn) if tp+fn else 0.0

        f1 = (
            2*precision*recall/(precision+recall)
            if precision+recall
            else 0.0
        )

        f1s.append(f1)

    return float(np.mean(f1s))

def fscr(truth,pred):
    truth = pd.Series(truth)
    pred = pd.Series(pred)

    invalid = truth.eq("REFUTED")

    return (
        float((pred[invalid]=="PROVEN").mean())
        if invalid.any()
        else np.nan
    )

def false_refutation_rate(truth,pred):
    truth = pd.Series(truth)
    pred = pd.Series(pred)

    valid = truth.eq("PROVEN")

    return (
        float((pred[valid]=="REFUTED").mean())
        if valid.any()
        else np.nan
    )

final_evaluation = pd.DataFrame()
final_metrics = pd.DataFrame()

if not human_final.empty:
    final_evaluation = (
        internal_manifest
        .merge(
            human_final,
            on="human_case_id",
            how="left"
        )
    )

    resolved = final_evaluation[
        final_evaluation.human_final_verdict.isin(
            LABELS
        )
    ].copy()

    rows = []

    for system,col in [
        ("SciGeoGuard_X","frozen_prediction"),
        ("Qwen2.5_7B","A_verdict"),
        ("Mistral_7B","B_verdict"),
        ("Generated_Reference","generated_expected")
    ]:
        rows.append({
            "system":system,
            "n_human_resolved":len(resolved),
            "accuracy_vs_human":
                exact_accuracy(
                    resolved.human_final_verdict,
                    resolved[col]
                ),
            "macro_f1_vs_human":
                macro_f1(
                    resolved.human_final_verdict,
                    resolved[col]
                ),
            "FSCR_vs_human":
                fscr(
                    resolved.human_final_verdict,
                    resolved[col]
                ),
            "false_refutation_rate_vs_human":
                false_refutation_rate(
                    resolved.human_final_verdict,
                    resolved[col]
                )
        })

    final_metrics = pd.DataFrame(rows)
    display(final_metrics)
else:
    print("System-vs-human metrics await final human labels.")

# Part XVI — Bootstrap 95% confidence intervals

Confidence intervals are calculated over resolved human-labelled cases.

In [ ]:
def bootstrap_metric(
    df,
    pred_col,
    metric_fn,
    B=3000,
    seed=SEED
):
    if len(df)==0:
        return np.nan,np.nan,np.nan

    rng = np.random.default_rng(seed)
    point = metric_fn(
        df.human_final_verdict,
        df[pred_col]
    )

    vals = []

    for _ in range(B):
        idx = rng.integers(
            0,
            len(df),
            len(df)
        )

        s = df.iloc[idx]

        val = metric_fn(
            s.human_final_verdict,
            s[pred_col]
        )

        if val == val:
            vals.append(val)

    if not vals:
        return point,np.nan,np.nan

    lo,hi = np.percentile(
        vals,
        [2.5,97.5]
    )

    return float(point),float(lo),float(hi)

ci_rows = []

if not final_evaluation.empty:
    resolved = final_evaluation[
        final_evaluation.human_final_verdict.isin(
            LABELS
        )
    ].copy()

    for system,col in [
        ("SciGeoGuard_X","frozen_prediction"),
        ("Qwen2.5_7B","A_verdict"),
        ("Mistral_7B","B_verdict"),
        ("Generated_Reference","generated_expected")
    ]:
        for metric_name,fn in [
            ("accuracy",exact_accuracy),
            ("macro_f1",macro_f1),
            ("FSCR",fscr),
            ("false_refutation_rate",false_refutation_rate)
        ]:
            p,lo,hi = bootstrap_metric(
                resolved,
                col,
                fn
            )

            ci_rows.append({
                "system":system,
                "metric":metric_name,
                "point":p,
                "ci95_low":lo,
                "ci95_high":hi
            })

ci_df = pd.DataFrame(ci_rows)

if len(ci_df):
    display(ci_df)

# Part XVII — Paired correctness comparison

A paired exact McNemar-style test is computed for SciGeoGuard-X versus each 7B LLM on the same human-labelled cases.

This tests whether one system is correct on significantly more of the discordant cases.

In [ ]:
def exact_binomial_two_sided(k,n):
    if n == 0:
        return np.nan

    k_small = min(k,n-k)

    prob = sum(
        math.comb(n,i)
        for i in range(k_small+1)
    ) / (2**n)

    return min(1.0, 2*prob)

paired_rows = []

if not final_evaluation.empty:
    resolved = final_evaluation[
        final_evaluation.human_final_verdict.isin(
            LABELS
        )
    ].copy()

    sgx_correct = (
        resolved.frozen_prediction ==
        resolved.human_final_verdict
    )

    for name,col in [
        ("Qwen2.5_7B","A_verdict"),
        ("Mistral_7B","B_verdict")
    ]:
        other_correct = (
            resolved[col] ==
            resolved.human_final_verdict
        )

        sgx_only = int(
            (sgx_correct & ~other_correct).sum()
        )

        other_only = int(
            (~sgx_correct & other_correct).sum()
        )

        discordant = sgx_only + other_only

        p = exact_binomial_two_sided(
            min(sgx_only,other_only),
            discordant
        )

        paired_rows.append({
            "comparison":f"SciGeoGuard_X vs {name}",
            "SciGeoGuard_only_correct":sgx_only,
            "other_only_correct":other_only,
            "discordant_cases":discordant,
            "exact_two_sided_p":p
        })

paired_df = pd.DataFrame(paired_rows)

if len(paired_df):
    display(paired_df)

# Part XVIII — Human disagreement analysis

Cases with A/B disagreement are retained as a separate qualitative dataset even after adjudication.

In [ ]:
disagreement_analysis = pd.DataFrame()

if not review_pair.empty:
    disagreement_analysis = (
        internal_manifest
        .merge(
            review_pair,
            on="human_case_id",
            how="inner"
        )
    )

    disagreement_analysis = disagreement_analysis[
        disagreement_analysis.A_human_verdict !=
        disagreement_analysis.B_human_verdict
    ].copy()

    print(
        "A/B disagreement cases:",
        len(disagreement_analysis)
    )

# Part XIX — Final figures

In [ ]:
FIGURES = []

if not final_metrics.empty:
    import matplotlib.pyplot as plt

    p = WORK/"figure_v7_accuracy_vs_human.png"

    plt.figure(figsize=(8,4))
    plt.bar(
        final_metrics.system,
        final_metrics.accuracy_vs_human
    )
    plt.ylabel("Accuracy vs final human verdict")
    plt.ylim(0,1.05)
    plt.xticks(rotation=20,ha="right")
    plt.tight_layout()
    plt.savefig(p,dpi=220)
    plt.show()

    FIGURES.append(p)

    p = WORK/"figure_v7_fscr_vs_human.png"

    plt.figure(figsize=(8,4))
    plt.bar(
        final_metrics.system,
        final_metrics.FSCR_vs_human
    )
    plt.ylabel("False Scientific Certification Rate vs human")
    plt.ylim(0,1.05)
    plt.xticks(rotation=20,ha="right")
    plt.tight_layout()
    plt.savefig(p,dpi=220)
    plt.show()

    FIGURES.append(p)

# Part XX — Export full internal validation bundle

In [ ]:
internal_manifest.to_csv(
    WORK/"INTERNAL_60_case_mapping_and_machine_context.csv",
    index=False
)

leakage_df.to_csv(
    WORK/"Reviewer_leakage_checks.csv",
    index=False
)

agreement_summary.to_csv(
    WORK/"Inter_rater_agreement.csv",
    index=False
)

if len(reviewer_confusion):
    reviewer_confusion.to_csv(
        WORK/"Reviewer_confusion_matrix.csv"
    )

adjudicator_blind.to_csv(
    WORK/"Adjudicator_BLIND.csv",
    index=False
)

human_final.to_csv(
    WORK/"Final_human_labels.csv",
    index=False
)

final_evaluation.to_csv(
    WORK/"Final_system_vs_human_case_results.csv",
    index=False
)

final_metrics.to_csv(
    WORK/"Final_system_vs_human_metrics.csv",
    index=False
)

ci_df.to_csv(
    WORK/"Final_metric_bootstrap_CI.csv",
    index=False
)

paired_df.to_csv(
    WORK/"Paired_correctness_tests.csv",
    index=False
)

disagreement_analysis.to_csv(
    WORK/"Human_reviewer_disagreement_analysis.csv",
    index=False
)

selection_summary = pd.DataFrame({
    "dimension":[
        "n_cases",
        "n_families",
        "BLIND",
        "DEVELOPMENT",
        "HELDOUT-FAMILY",
        "reference_PROVEN",
        "reference_REFUTED",
        "reference_UNPROVEN"
    ],
    "value":[
        len(sample60),
        sample60.family.nunique(),
        int((sample60.split=="BLIND").sum()),
        int((sample60.split=="DEVELOPMENT").sum()),
        int((sample60.split=="HELDOUT-FAMILY").sum()),
        int((sample60.generated_expected=="PROVEN").sum()),
        int((sample60.generated_expected=="REFUTED").sum()),
        int((sample60.generated_expected=="UNPROVEN").sum())
    ]
})

selection_summary.to_csv(
    WORK/"INTERNAL_selection_summary.csv",
    index=False
)

README = f"""
SciGeoGuard-X Final Blinded Human Validation v7
================================================

Sample:
60 workflows selected deterministically from the completed v6 experiment.

Sampling strata:
- 20 LLM consensus vs SciGeoGuard conflicts
- 10 one-LLM-matches-SciGeoGuard conflicts
- 10 all-system agreement controls
- 10 UNPROVEN/missing-evidence stress cases
- 10 stratified random controls

Reviewer protection:
- neutral HV60 IDs
- original family-bearing IDs hidden
- machine verdicts hidden
- generated reference hidden
- Reviewer A/B orders randomized independently

Recommended protocol:
1. Reviewer A and Reviewer B work independently.
2. Import completed A/B files.
3. Send Adjudicator_BLIND.csv to a third reviewer for disagreements.
4. Import Adjudicator_completed.csv.
5. Report metrics against final human labels.

Do not call generated labels or LLM outputs human ground truth.

Created UTC:
{datetime.now(timezone.utc).isoformat()}
"""

(WORK/"README_v7.txt").write_text(
    README,
    encoding="utf-8"
)

full_zip = shutil.make_archive(
    str(ROOT/"SciGeoGuard_X_v7_Final_Blinded_Human_Validation"),
    "zip",
    root_dir=WORK
)

print("Reviewer ZIP:", reviewer_zip)
print("Full internal ZIP:", full_zip)

try:
    from google.colab import files
    files.download(reviewer_zip)
except Exception:
    pass

# Final interpretation

The key distinction in v7 is that the final accuracy comparison is no longer based only on generated benchmark labels.

Once two independent reviewers and blind adjudication are complete, the paper can report:

- human inter-rater agreement;
- SciGeoGuard-X accuracy / Macro-F1 / FSCR against human labels;
- Qwen-7B accuracy / Macro-F1 / FSCR against the same human labels;
- Mistral-7B accuracy / Macro-F1 / FSCR against the same human labels;
- generated-reference agreement with humans;
- confidence intervals;
- paired system comparisons;
- qualitative analysis of human/model disagreements.

This is the validation stage required for the strongest empirical claims.